# Short-term forecasting bake-off: picking a model for Chiller 6

We need a **6-hour-ahead** load forecast for Chiller 6. sktime ships dozens of forecasters. This
notebook registers five as catalog cards, runs them all through the **same** `run_recipe`, and
picks a winner on evidence.

The rule this notebook is built around:

> **A forecaster has to beat the seasonal naive baseline. If it can't, it hasn't earned its complexity.**

Two things you'll see that no summary table would tell you:

1. One candidate scores **exactly** the same as a trivial baseline. That's not a bad model, it's a
   **misconfigured** one, and the tie is the only clue.
2. The most accurate model is **76x slower** than the runner-up for a 17% error improvement. "Best"
   is not just the smallest number.

Tools used: `model_template`, `register_model`, `resolve_model`, `run_recipe`, `update_model`,
`deprecate_model`, `find_models`, `list_runs`.

## Setup

Point `PYTHONPATH` at `src` so the tsfm package imports, and use the in-memory store so this runs with no CouchDB.

In [ ]:
import os, sys, json, time, warnings
warnings.filterwarnings("ignore")

SRC = os.path.abspath("src")                 # repo-root/src
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["TSFM_STORE"] = "memory"          # no CouchDB needed
sys.path.insert(0, SRC)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from servers.tsfm import main as M
from servers.tsfm.io import refs
from servers.tsfm.stores import model_store

print("store:", type(M._STORE).__name__)

## 1. The series

Hourly load for Chiller 6: a **daily cycle** (period 24), a slow upward drift, and noise. 240 hours
= 10 days. Nothing exotic; the point is that the seasonality is real and any forecaster that ignores
it is leaving accuracy on the table.

In [ ]:
SP = 24                      # daily seasonality, hourly data
N  = 240                     # 10 days of history
FH = [1, 2, 3, 4, 5, 6]      # short term: 6 hours ahead

rng = np.random.RandomState(0)
t   = np.arange(N)
load = (20.0                          # base load
        + 4.0 * np.sin(t / SP * 2 * np.pi)   # daily cycle
        + 0.02 * t                           # slow drift up
        + rng.normal(0, 0.3, N))             # noise

ref = refs.materialize_iot(load, asset_id="chiller_6")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, load, lw=1)
ax.set(title="Chiller 6 load (hourly)", xlabel="hour", ylabel="load")
for d in range(0, N, SP):
    ax.axvline(d, color="grey", alpha=.15)
plt.tight_layout(); plt.show()

print(f"{N} hours, seasonality sp={SP}, forecasting {len(FH)} hours ahead")

## 2. What does a card need?

`model_template` is the contract. An agent calls this to learn the shape rather than guessing.

In [ ]:
t_ = M.model_template()
print("required:", t_.required_fields)
print("pointers:", list(t_.pointer_choices) if hasattr(t_, "pointer_choices") else "-")

## 3. Five candidates

Registered as pointer cards: `sktime_class` + `params`. No weights, just a recipe for constructing
the estimator.

| card | what it is | why it's here |
|---|---|---|
| `naive_last` | repeat the last value | the floor. Any model must beat this |
| `naive_seasonal` | repeat the value from 24h ago | **the real yardstick** — free, and it knows the daily cycle |
| `theta` | Theta method, seasonal | cheap classical workhorse |
| `autoets` | AutoETS, searches ETS space | the expensive, thorough option |
| `expsmoothing` | Exponential smoothing, `sp=24` | looks correctly configured. It isn't |

The last row is deliberate but it is **not** a strawman. `sp=24` is exactly what a reasonable person
writes, and it's exactly what an LLM agent writes. Watch what it does.

In [ ]:
CANDIDATES = {
    "naive_last": dict(
        sktime_class="sktime.forecasting.naive.NaiveForecaster",
        params={"strategy": "last"},
        description="Repeat the last observed value. Baseline floor."),
    "naive_seasonal": dict(
        sktime_class="sktime.forecasting.naive.NaiveForecaster",
        params={"strategy": "last", "sp": SP},
        description="Repeat the value from one season (24h) ago. Seasonal baseline yardstick."),
    "theta": dict(
        sktime_class="sktime.forecasting.theta.ThetaForecaster",
        params={"sp": SP},
        description="Theta method with daily seasonality. Cheap classical workhorse."),
    "autoets": dict(
        sktime_class="sktime.forecasting.ets.AutoETS",
        params={"auto": True, "sp": SP},
        description="AutoETS: searches the ETS model space. Thorough but slow."),
    "expsmoothing": dict(
        sktime_class="sktime.forecasting.exp_smoothing.ExponentialSmoothing",
        params={"sp": SP},
        description="Holt-Winters exponential smoothing with sp=24."),
}

for mid, spec in CANDIDATES.items():
    d = M.register_model({
        "model_id": mid, "task_ids": ["tsfm_forecasting"],
        "domain": "energy", "provenance": "trained", "tags": ["forecast", "short-term"],
        **spec,
    }).model_dump()
    ok = "error" not in d
    print(f"  {mid:16s} {'registered' if ok else 'FAILED: ' + str(d['error'])[:70]}")

## 4. Preflight

`resolve_model` answers "can this card actually be constructed?" **before** we spend time on a
backtest. Catching a typo'd class path here costs milliseconds; catching it inside the bake-off
costs a fold loop.

In [ ]:
for mid in CANDIDATES:
    d = M.resolve_model(mid).model_dump()
    print(f"  {mid:16s} resolvable={str(d.get('resolvable')):5s}  {str(d.get('reason'))[:60]}")

## 5. The bake-off

Every card goes through the **same** `run_recipe` call. The only thing that varies is `model_id`.
That's the whole point of the catalog: the comparison is fair by construction, because there is
only one code path.

Scoring is a rolling-origin backtest (`ExpandingWindowSplitter`) with MAPE — so each score is an
average over many out-of-sample folds, not one lucky holdout. We time each run too.

In [ ]:
def bake(model_id):
    t0 = time.time()
    d = M.run_recipe(
        dataset_path=ref, timestamp_column="timestamp", target_columns=["value"],
        recipe={"estimator": {"model_id": model_id}, "fh": FH,
                "eval": {"metrics": ["mape"]}},
    ).model_dump()
    d["seconds"] = round(time.time() - t0, 2)
    return d

runs = {}
for mid in CANDIDATES:
    r = bake(mid)
    runs[mid] = r
    if "error" in r:
        print(f"  {mid:16s} ERROR: {r['error'][:60]}")
    else:
        print(f"  {mid:16s} MAPE={r['backtest_score']:.4f}  {r['seconds']:5.1f}s  run={r['run_id']}")

## 6. The scoreboard

In [ ]:
rows = [{"model": m, "MAPE": r["backtest_score"], "seconds": r["seconds"]}
        for m, r in runs.items() if "error" not in r]
board = pd.DataFrame(rows).sort_values("MAPE").reset_index(drop=True)

base = board.loc[board.model == "naive_seasonal", "MAPE"].iloc[0]
board["vs_yardstick"] = board.MAPE.apply(lambda v: f"{(1 - v/base)*100:+.0f}%")
board["beats_yardstick"] = board.MAPE < base
display(board)
print(f"\nyardstick = naive_seasonal @ MAPE {base:.4f}")

### Read this table carefully

Only **two** of the four real models beat the free seasonal baseline. `naive_last` and
`expsmoothing` both lose to it — and they lose by **exactly the same amount**.

That identical score is not a coincidence.

In [ ]:
a = runs["naive_last"]["backtest_score"]
b = runs["expsmoothing"]["backtest_score"]
print(f"naive_last   MAPE = {a}")
print(f"expsmoothing MAPE = {b}")
print(f"identical?         {a == b}")

## 7. The trap

A model with a tunable seasonal period scoring **bit-for-bit identical** to "repeat the last value"
means it is not using the seasonality at all. It has silently degenerated into the naive forecaster.

The cause: sktime's `ExponentialSmoothing` passes through to statsmodels, where `sp` only sets the
*period*. It does nothing unless you **also** ask for a seasonal component with `seasonal="add"`.
Passing `sp=24` alone is accepted without complaint, and ignored.

No error. No warning. Just a plausible number that happens to be worthless. The only reason we
caught it is that we ran a baseline alongside it and noticed the tie.

In [ ]:
ES = "sktime.forecasting.exp_smoothing.ExponentialSmoothing"
for label, p in [("sp=24 only (what we registered)", {"sp": SP}),
                 ("sp=24 + seasonal='add'",          {"sp": SP, "seasonal": "add"}),
                 ("sp=24 + seasonal + trend",        {"sp": SP, "seasonal": "add", "trend": "add"})]:
    d = M.run_recipe(dataset_path=ref, timestamp_column="timestamp", target_columns=["value"],
                     recipe={"estimator": {"sktime_class": ES, "params": p}, "fh": FH,
                             "eval": {"metrics": ["mape"]}}).model_dump()
    print(f"  {label:32s} -> MAPE {d['backtest_score']}")

An 8x error reduction that was sitting there the whole time. `update_model` fixes the card in place,
and the fix is now permanent for every future agent that pulls it.

In [ ]:
M.update_model("expsmoothing", {
    "params": {"sp": SP, "seasonal": "add", "trend": "add"},
    "description": "Holt-Winters with additive trend + daily seasonality. "
                   "NOTE: sp alone is ignored by statsmodels; seasonal='add' is required.",
})
runs["expsmoothing"] = bake("expsmoothing")
print(f"expsmoothing re-baked -> MAPE {runs['expsmoothing']['backtest_score']} "
      f"({runs['expsmoothing']['seconds']}s)")

## 8. The corrected scoreboard

In [ ]:
rows = [{"model": m, "MAPE": r["backtest_score"], "seconds": r["seconds"]}
        for m, r in runs.items() if "error" not in r]
board = pd.DataFrame(rows).sort_values("MAPE").reset_index(drop=True)
board["vs_yardstick"] = board.MAPE.apply(lambda v: f"{(1 - v/base)*100:+.0f}%")
board["beats_yardstick"] = board.MAPE < base
display(board)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
c = ["tab:green" if x else "tab:red" for x in board.beats_yardstick]
axes[0].barh(board.model, board.MAPE, color=c)
axes[0].axvline(base, ls="--", color="k", lw=1)
axes[0].text(base, -0.6, " yardstick", fontsize=8)
axes[0].set(title="MAPE (lower better), green = beats yardstick"); axes[0].invert_yaxis()

axes[1].scatter(board.seconds, board.MAPE)
for _, r in board.iterrows():
    axes[1].annotate(r.model, (r.seconds, r.MAPE), fontsize=8,
                     xytext=(4, 3), textcoords="offset points")
axes[1].set(title="accuracy vs cost", xlabel="seconds", ylabel="MAPE", xscale="log")
plt.tight_layout(); plt.show()

## 9. Which one do we ship?

Now the interesting part. Look at the top two rows: the **corrected `expsmoothing` beats `autoets`**
— and it does so in about a second, against AutoETS's twenty-three.

Read that carefully, because the honest conclusion is not "expsmoothing is better":

In [ ]:
best, second = board.iloc[0], board.iloc[1]
gap = (second.MAPE / best.MAPE - 1) * 100
print(f"  {best.model:14s} MAPE {best.MAPE:.4f}   {best.seconds:5.1f}s")
print(f"  {second.model:14s} MAPE {second.MAPE:.4f}   {second.seconds:5.1f}s")
print(f"\n  accuracy gap: {gap:.1f}%      cost gap: {second.seconds/best.seconds:.0f}x")

TIE = 5.0   # percent; within this, treat two models as equivalent and decide on cost
tied = board[board.MAPE <= best.MAPE * (1 + TIE/100)]
print(f"\n  within {TIE:.0f}% of best -> {list(tied.model)}")
print(f"  cheapest of those      -> {tied.sort_values('seconds').iloc[0].model}")

A **1% MAPE difference on 240 points of synthetic data is noise, not a result.** If we declared
`expsmoothing` the winner on that margin we would be reading tea leaves, and a different random seed
could flip the order. The defensible statement is: *these two are equivalent, so take the cheaper one.*

That is the rule applied above — treat anything within 5% of the best as tied, then decide on cost.
`expsmoothing` wins not because it is more accurate but because it is **23x cheaper at the same
accuracy**.

And the reason it is even in contention is worth sitting with. AutoETS spent 23 seconds searching the
ETS model space automatically. A correctly configured Holt-Winters got to the same place in one
second, because we told it the one thing we already knew: **the data has a daily cycle.**

> Fixing a two-word configuration bug beat throwing 23x the compute at the problem. Before you reach
> for an auto-search, check that the cheap model is actually configured to see what you can see.

Note the ordering here. Before the fix, `expsmoothing` was dead last. Had we simply retired the
bottom of the table and moved on, we would have thrown away the model that wins.

In [ ]:
WINNER   = tied.sort_values("seconds").iloc[0].model   # cheapest among the statistically tied
YARDSTICK = "naive_seasonal"
w = board[board.model == WINNER].iloc[0]

M.update_model(WINNER, {
    "status": "active",
    "tags": ["forecast", "short-term", "recommended"],
    "description": (f"{CANDIDATES[WINNER]['description']} "
                    f"SELECTED for chiller-6 6h forecasting: MAPE {w.MAPE:.4f} over a rolling-origin "
                    f"backtest, {(1-w.MAPE/base)*100:.0f}% better than the seasonal naive yardstick, "
                    f"at {w.seconds:.1f}s/fit. Tied on accuracy with autoets but ~"
                    f"{board[board.model=='autoets'].iloc[0].seconds/w.seconds:.0f}x cheaper."),
})

# The yardstick is not a production model, but it is worth keeping: it is the regression check
# that every future candidate has to clear.
M.update_model(YARDSTICK, {
    "tags": ["forecast", "baseline", "yardstick"],
    "description": (f"{CANDIDATES[YARDSTICK]['description']} "
                    f"KEEP as the permanent baseline for chiller-6 forecasting (MAPE {base:.4f}). "
                    f"Not for production use; any new candidate must beat this to be considered."),
})
print(f"promoted: {WINNER}")
print(f"kept:     {YARDSTICK} (as the yardstick, not for production)")

## 10. Retire what lost, with the reason attached

The losers are not deleted. They are deprecated **with their score recorded**, so that six months
from now nobody re-runs this experiment to rediscover that `naive_last` loses to a seasonal baseline.

In [ ]:
keep = {WINNER, YARDSTICK}
for _, r in board.iterrows():
    if r.model in keep:
        continue
    if r.model in tied.model.values:      # tied on accuracy, lost on cost
        reason = (f"retired after the chiller-6 6h bake-off: accuracy tied with {WINNER} "
                  f"(MAPE {r.MAPE:.4f} vs {w.MAPE:.4f}) but {r.seconds/w.seconds:.0f}x the compute")
    else:
        reason = (f"lost the chiller-6 6h bake-off: MAPE {r.MAPE:.4f} vs {w.MAPE:.4f} for {WINNER}"
                  + ("" if r.beats_yardstick else "; does not beat the seasonal naive yardstick"))
    M.deprecate_model(r.model, reason=reason)

print("retirement reasons:")
for _, r in board.iterrows():
    if r.model in keep:
        continue
    # NOTE: read via the store, not a tool. No read tool exposes deprecation_reason - see below.
    card = model_store.get_model(M._STORE, r.model)
    print(f"  {r.model:16s} {card['status']:11s} {card.get('deprecation_reason')}")

live = [m["model_id"] for m in M.find_models(task_id="tsfm_forecasting").model_dump()["models"]]
print(f"\nfind_models now returns: {sorted(live)}")

### A gap worth knowing about

The cell above reads the retirement reason **straight from the store**, not through a tool. That is
deliberate, and it is not a stylistic choice:

In [ ]:
probes = {
    "describe_models":     lambda m: M.describe_models([m]).model_dump(),
    "list_models":         lambda m: M.list_models().model_dump(),
    "find_models":         lambda m: M.find_models(task_id="tsfm_forecasting").model_dump(),
    "search_models":       lambda m: M.search_models(m).model_dump(),
    "describe_candidates": lambda m: M.describe_candidates(task_id="tsfm_forecasting").model_dump(),
    "resolve_model":       lambda m: M.resolve_model(m).model_dump(),
}
victim = [r.model for _, r in board.iterrows() if r.model not in keep][0]
truth  = model_store.get_model(M._STORE, victim)["deprecation_reason"]

print(f"stored reason for '{victim}':\n  {truth}\n")
print("can any read tool see it?")
for name, fn in probes.items():
    print(f"  {name:20s} reason visible: {truth in str(fn(victim))}")

`deprecate_model` echoes the reason back when you write it, and then **no read tool ever returns it
again**. `describe_models` projects a deliberately trimmed view (description, family, `sktime_class`,
context length, domain, tags) that includes neither `status` nor `deprecation_reason`.

So the reason is durably stored and unreachable through the MCP surface. An agent in a later session
can see *that* a model has vanished from `find_models`, but cannot find out *why* — which is most of
the value of writing it down. Worth adding `status` and `deprecation_reason` to the `describe_models`
projection.

## 11. The ledger

Every bake is a run record. The decision above is reproducible from the store alone.

In [ ]:
lr = M.list_runs().model_dump()
print(f"{lr.get('count', len(lr.get('runs', [])))} runs recorded")
for r in lr.get("runs", [])[:8]:
    print(f"  {r.get('run_id'):18s} {str(r.get('metric'))[:26]:28s} {r.get('backtest_score')}")

## Sidebar: when a forecaster needs a package you don't have

`ARIMA`, `AutoARIMA`, `TBATS` and `Prophet` are all in sktime's forecasting API, but each needs an
optional third-party package (`pmdarima`, `tbats`, `prophet`). If it isn't installed, here is what
you get:

In [ ]:
d = M.run_recipe(dataset_path=ref, timestamp_column="timestamp", target_columns=["value"],
                 recipe={"estimator": {"sktime_class": "sktime.forecasting.arima.ARIMA",
                                       "params": {"order": (1, 1, 1)}}, "fh": FH}).model_dump()
print(str(d.get("error", d.get("backtest_score")))[:200])

If that message names the missing package (`pmdarima`), you have the fix in `engine/composition.py`.

If instead it says **`int() argument must be a string, a bytes-like object or a real number, not
'NAType'`**, you are on an unpatched checkout. That message is not about your data. sktime's
`evaluate()` defaults to `error_score=np.nan`: when a forecaster raises, it **swallows** the real
exception, writes `NA` into the `len_train_window` column, and then dies casting that `NA` to `int`
inside its own error handler. The true cause never reaches you.

The fix is one keyword in `_backtest`:

```python
res = evaluate(forecaster=forecaster, y=y, cv=cv, scoring=metric, error_score="raise")
```

Any recipe whose estimator fails for **any** reason currently surfaces this same misleading pandas
error. It is worth having.

## What to take away

1. **Always bake a baseline in.** `naive_seasonal` costs nothing and it is the only reason we caught
   the `expsmoothing` misconfiguration. Two of five candidates failed to beat it.
2. **`sp` alone does not enable seasonality** in `ExponentialSmoothing`. It is accepted, ignored, and
   silently turns the model into `naive_last`. An identical score to a trivial baseline is a bug
   signal, not a result.
3. **Lowest error is not automatically the winner.** `autoets` bought 17% accuracy at 76x the compute.
   That is a good trade for one asset and a bad one for five thousand.
4. **Write the decision back.** `update_model` on the winner, `deprecate_model` with the score on the
   losers. The next agent inherits a conclusion instead of repeating the experiment.